# Replacing the expert ranking

**Paper output being replaced:** Table 10. Each industry gets a score (low = more polluting) and is ranked within its industrial area.
`score = Σ criterion_weight(C1..C10) × local_weight(industry, criterion)`. The experts produced the local weights (Table 6 ratings, then fuzzy AHP).

Both methods start from the same **per-industry summary**: one row per industry, one column per paper criterion, all computed from lab data.

| Criterion | Summary value | Worse when |
|---|---|---|
| C1 Effluent discharge | mean flow (MLD) | higher |
| C2 ETP/CETP efficiency | mean (inlet − outlet) / inlet | lower |
| C3 / C4 Critical parameters, pre- / post-monsoon | pollutants over the CPCB limit per sampling event | higher |
| C5 Effluent type | industry type → paper's 1–9 scale (Table 4) | lower |
| C6 / C7 River impact, pre- / post-monsoon | downstream − upstream, percentile within pollutant | higher |
| C8 / C9 Groundwater impact, pre- / post-monsoon | near well − control well, percentile within pollutant | higher |
| C10 Public health | Σ toxicity × amount over the limit, per event | higher |

**A. Learn the experts' scores.** Ridge regression: summary → paper score (normalised within area). Experts are needed once, for the training labels. Tested leave-one-out.

**B. Compute the criteria directly.** Rank industries within their area on each criterion, scale ranks to sum to 1 (the shape of the paper's local weights), then apply the paper's own weighted sum. No expert rates an industry. "Bad" comes from legal limits (C3, C4, C10), physical direction (C1, C2, C6–C9) and value choices set once (C5 scale, toxicity factors, criterion weights).

There's no true answer to check against. A is scored against expert scores; B can only be *compared* with them.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
import wqml

CRITERIA = [f"C{i}" for i in range(1, 11)]
WORSE_IF_HIGH = {c: c not in ("C2", "C5") for c in CRITERIA}
# Paper Table 4 C5 scale (9 = less organic ... 1 = hazardous). PLACEHOLDER mapping: confirm with domain experts.
C5_CLASS = {"tannery": 1, "steel": 1, "cetp": 3, "drain": 3, "textile": 3,
            "food": 5, "meat": 5, "oil": 5, "paper_pulp": 5}
W = pd.read_csv(wqml.PAPER / "criteria.csv").set_index("criterion")["weight"]  # paper criterion weights
W

## 1. Data
Order: your data (`data/measurements.csv`), a lab sheet (`data/samples.csv`), then **dummy** data.

In [ ]:
def dummy(seed=0):
    """Schema-identical FAKE data. Each industry's pollution is scaled by how bad the paper's
    experts scored it (area mean / fuzzy score), plus noise. Good agreement below is EXPECTED by
    construction; it proves the code works, not the method."""
    rng = np.random.default_rng(seed)
    raw = wqml.synthetic(seed=seed)
    p = wqml.paper_ranking().reset_index()
    sev = p.groupby("industrial_area")["fuzzy_weight"].transform("mean") / p["fuzzy_weight"]
    k = raw["source_id"].map(pd.Series(sev.to_numpy() * rng.lognormal(0, 0.3, len(p)), index=p["source_id"]))
    raw["outlet"] *= k
    raw["downstream"] = raw["upstream"] + (raw["downstream"] - raw["upstream"]) * k
    raw["near_well"] = raw["control_well"] + (raw["near_well"] - raw["control_well"]) * k
    raw["discharge_flow_mld"] *= k ** 0.5
    return raw

DATA, SAMPLES, STATIONS = Path("data/measurements.csv"), Path("data/samples.csv"), Path("data/stations.csv")
DUMMY = False
if DATA.exists():
    raw = pd.read_csv(DATA)
elif SAMPLES.exists():
    raw = wqml.reshape(pd.read_csv(SAMPLES), pd.read_csv(STATIONS if STATIONS.exists() else wqml.PAPER / "stations.csv"))
else:
    raw, DUMMY = dummy(), True
    print("DUMMY data: results show the code works, nothing about the real industries.")
df = wqml.prepare(raw)
df.shape

## 2. Per-industry summary (shared by A and B)

In [ ]:
def summarise(df):
    d = df.copy()
    # event = one sampling round: date, else season (paper rows are undated), else one 'undated' round
    d["event"] = d["sample_date"].dt.strftime("%Y-%m-%d").fillna(d["season"]).fillna("undated")
    undated = d["season"].isna()  # paper text rows: no date or season -> count in both seasons
    pre, post = undated | d["season"].eq("pre_monsoon"), undated | d["season"].eq("post_monsoon")
    every = pd.Series(True, index=d.index)
    d["excess"] = (d["exceedance_ratio"] - 1).clip(lower=0) * d["pollutant"].map(wqml.TOXICITY).fillna(1)
    for c in ["river_change", "gw_change"]:  # units differ by pollutant -> percentile within pollutant
        d[c] = d.groupby("pollutant")[c].rank(pct=True)
    eff = ((d["inlet"] - d["outlet"]) / d["inlet"]).where(d["inlet"].gt(0) & d["pollutant"].ne("pH"))

    def per_event(mask, col):  # sum over pollutants within one sampling event, then mean over events
        return d[mask].groupby(["source_id", "event"])[col].sum(min_count=1).groupby(level=0).mean()

    s = pd.DataFrame({
        "C1": d.groupby("source_id")["discharge_flow_mld"].mean(),
        "C2": eff.groupby(d["source_id"]).mean(),
        "C3": per_event(pre, "violation"),
        "C4": per_event(post, "violation"),
        "C6": d[pre].groupby("source_id")["river_change"].mean(),
        "C7": d[post].groupby("source_id")["river_change"].mean(),
        "C8": d[pre].groupby("source_id")["gw_change"].mean(),
        "C9": d[post].groupby("source_id")["gw_change"].mean(),
        "C10": per_event(every, "excess"),
    })
    s = d.groupby("source_id")[["industrial_area", "industry_type"]].first().join(s)
    s["C5"] = s["industry_type"].map(C5_CLASS)
    return s[["industrial_area", "industry_type"] + CRITERIA]

summary = summarise(df)
summary.round(3)

## 3. Labels: the paper's fuzzy scores
Table 10 scores are relative within each area, so raw values can't be compared across areas. Target = score ÷ area mean (1 = average for its area, below 1 = more polluting).

In [ ]:
paper = wqml.paper_ranking().reset_index()
paper["target"] = paper["fuzzy_weight"] / paper.groupby("industrial_area")["fuzzy_weight"].transform("mean")
labels = paper.set_index("source_id")[["fuzzy_weight", "target"]]

def agreement(t, col, ref="target"):
    """How closely a score reproduces the reference ranking: Spearman overall and per area,
    and the share of areas where both name the same most-polluting industry."""
    by_area = t.groupby("industrial_area")
    out = {"spearman_all": spearmanr(t[col], t[ref]).statistic}
    out |= {f"spearman_{a}": spearmanr(g[col], g[ref]).statistic for a, g in by_area if len(g) > 2}
    out["same_worst_industry"] = by_area.apply(lambda g: g[col].idxmin() == g[ref].idxmin()).mean()
    return pd.Series(out).round(3)
labels.head()

## 4. Method A: learn the experts' scores
Ridge regression, a small linear model with a penalty that shrinks weak effects. Chosen because there are only 33 labelled industries. Each prediction comes from a model trained on the other 32 (leave-one-out).

In [ ]:
train = summary.join(labels, how="inner")
X, y = train[CRITERIA], train["target"]
model_a = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                        FunctionTransformer(np.arcsinh), StandardScaler(),
                        RidgeCV(alphas=np.logspace(-2, 3, 30)))
train["A_score"] = cross_val_predict(model_a, X, y, cv=LeaveOneOut())
print(f"n = {len(train)} | MAE model {abs(train.A_score - y).mean():.3f} vs always-average {abs(1 - y).mean():.3f}")
agreement(train, "A_score")

In [ ]:
# Which criteria the learned expert behaviour leans on (standardised coefficients; + = less polluting)
model_a.fit(X, y)
pd.Series(model_a[-1].coef_, index=CRITERIA).round(3).sort_values()

## 5. Method B: compute the criteria directly
Per criterion: rank industries within their area (worse = lower), scale ranks to sum to 1 like the paper's local weights (missing = neutral), then apply Eq. 10 with the paper's criterion weights.

In [ ]:
def method_b(s, weights=W):
    area = s["industrial_area"]
    local = pd.DataFrame(index=s.index)
    for c in CRITERIA:
        good = -s[c] if WORSE_IF_HIGH[c] else s[c]
        r = good.groupby(area).rank()
        r = r.fillna((area.map(area.value_counts()) + 1) / 2)  # not measured -> middle rank
        local[c] = r / r.groupby(area).transform("sum")
    out = s[["industrial_area"]].copy()
    out["B_score"] = (local * weights[CRITERIA]).sum(axis=1)
    out["B_score"] /= out.groupby("industrial_area")["B_score"].transform("mean")  # same scale as target
    out["B_rank"] = out.groupby("industrial_area")["B_score"].rank(method="min").astype(int)  # 1 = most polluting
    return out, local

b, local = method_b(summary)
train = train.join(b[["B_score", "B_rank"]])
agreement(train, "B_score")

In [ ]:
def explain(source, s=summary, local=local):
    """Why an industry scored as it did: each criterion's weighted contribution relative to a
    neutral (average) industry in its area. Negative = pulled it towards 'more polluting'."""
    n = (s["industrial_area"] == s.loc[source, "industrial_area"]).sum()
    contrib = (local.loc[source] - 1 / n) * W[CRITERIA]
    return pd.DataFrame({"measured": s.loc[source, CRITERIA], "vs_neutral": contrib.round(4)})

worst = train.groupby("industrial_area")["B_score"].idxmin()
print("B's most polluting industry per area:", worst.to_dict())
explain(worst.iloc[0])

## 6. Fuzzy vs A vs B

In [ ]:
train["fuzzy_rank"] = train.groupby("industrial_area")["fuzzy_weight"].rank(method="min").astype(int)
train["A_rank"] = train.groupby("industrial_area")["A_score"].rank(method="min").astype(int)
side = train[["industrial_area", "fuzzy_rank", "A_rank", "B_rank", "target", "A_score", "B_score"]]
display(side.sort_values(["industrial_area", "fuzzy_rank"]).round(3))
pd.DataFrame({"A (learn experts)": agreement(train, "A_score"), "B (compute)": agreement(train, "B_score"),
              "A vs B": agreement(train, "A_score", ref="B_score")})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for a, col in zip(ax, ["A_score", "B_score"]):
    a.scatter(train[col], train["target"])
    a.axline((1, 1), slope=1, ls="--", c="grey")
    a.set_xlabel(col); a.set_title(col.split("_")[0] + (" (dummy)" if DUMMY else ""))
ax[0].set_ylabel("paper score / area mean")
plt.tight_layout()

## 7. Real check: B on the paper's own published rows
Only 8 industries have published measurements, two per area. For each area: does B pick the same more-polluting industry of the pair as the experts? Weak evidence: the authors printed these numbers *because* these were the best and worst industries.

In [ ]:
real = wqml.prepare(pd.read_csv(wqml.PAPER / "measurements.csv"))
rs = summarise(real)
rb, _ = method_b(rs)
rb = rb.join(labels)
pairs = rb.groupby("industrial_area").apply(lambda g: pd.Series({
    "industries": ", ".join(g.index), "B_worse": g["B_score"].idxmin(), "experts_worse": g["fuzzy_weight"].idxmin()}))
pairs["agree"] = pairs["B_worse"] == pairs["experts_worse"]
display(rs[CRITERIA].round(2))
pairs

## 8. B on open data: US EPA ECHO
Real repeated monthly discharge reports (2019–2024) from US textile, paper, meat and tannery plants. There are no expert scores, so only B can run. Industries are ranked within their sector (sector stands in for industrial area); the limit is each facility's own permit limit. Only 5 facilities have upstream/downstream data (C6/C7); none have groundwater data.

In [ ]:
echo = wqml.prepare(pd.read_csv(wqml.ECHO / "echo_dmr.csv"))
echo["industrial_area"] = echo["industry_type"]
es = summarise(echo)
eb, elocal = method_b(es)
names = pd.read_csv(wqml.ECHO / "facilities.csv").set_index("npdes_id")["name"]
eb.join(names).join(es[CRITERIA].round(2)).sort_values(["industrial_area", "B_rank"])

In [ ]:
explain(eb["B_score"].idxmin(), s=es, local=elocal)

## Before using either on real Unnao data
- [ ] Lab data for all 33 industries (`data/samples.csv` + station register), so both methods cover every labelled industry
- [ ] A: leave-one-out agreement clearly better than always-average; check the coefficients make physical sense
- [ ] B: `C5_CLASS`, `wqml.TOXICITY` and criterion weights confirmed by domain experts
- [ ] Look at every industry where A, B and the paper disagree, using `explain()`
- [ ] C6–C9 need upstream/downstream and paired wells; until then they are neutral in B and imputed in A